# Temporal sensitivity of the Dirichlet model (Reviewer 76A)

Puts together the per-window fit statistics from `11-Dirichlet_regression_sensitivity.R` (5 weekday windows × 2 elections) and quantifies how much the mobile-service signal adds to the model under each window. Pure pandas — reads the CSVs in `data/dirichlet_sensitivity/`, so it runs in any kernel.

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 12

import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f"{WORKING_DIR}/data"
IMG_DIR = f"{WORKING_DIR}/images"
SENS_DIR = f"{DATA_DIR}/dirichlet_sensitivity"

WINDOWS = ["20_07", "07_20", "07_13", "13_20", "full"]
WINDOW_LABELS = {
    "20_07": "20-07 (residential)",
    "07_20": "07-20 (daytime)",
    "07_13": "07-13 (morning)",
    "13_20": "13-20 (afternoon)",
    "full": "full day",
}
YEARS = ["2019", "2024"]

## Load the per-window fit statistics

In [ ]:
# Load the 10 per-window fit-stat tables produced by 11-Dirichlet_regression_sensitivity.R
rows = []
for f in sorted(glob.glob(f"{SENS_DIR}/*/*/df_parameters_fit.csv")):
    m = re.search(r"/(\d{4})/([^/]+)/df_parameters_fit", f)
    d = pd.read_csv(f)
    d["year"], d["window"] = m.group(1), m.group(2)
    rows.append(d)
fit = pd.concat(rows, ignore_index=True)
assert fit.isna().sum().sum() == 0, "some fits failed (NA present)"
print("models:", list(fit["model"].unique()))
print("combos:", fit.groupby(["year", "window"]).ngroups, "(expect 10)")
fit.head()

## Mobile-service contribution per window (all − socioeconomic)

In [ ]:
# Mobile-service contribution = full model ("all") vs the window-invariant
# socioeconomic baseline, in the same likelihood criteria as the main Table 5.
ALL, SOCIO = "income_unemployment_pop_apps", "income_unemployment_pop"


def val(year, window, model, col):
    s = fit[(fit.year == year) & (fit.window == window) & (fit.model == model)][col]
    return float(s.iloc[0]) if len(s) else np.nan


rec = []
for year in YEARS:
    for w in WINDOWS:
        row = {"year": year, "window": w}
        for col in ["LogLik", "AIC", "BIC"]:
            row[f"d{col}"] = val(year, w, ALL, col) - val(year, w, SOCIO, col)
        row["all_AIC"] = val(year, w, ALL, "AIC")
        row["socio_AIC"] = val(year, w, SOCIO, "AIC")
        rec.append(row)
delta = pd.DataFrame(rec)


def show(metric, title):
    t = delta.pivot_table(index="window", columns="year", values=metric).reindex(
        WINDOWS
    )
    t.index = [WINDOW_LABELS[w] for w in WINDOWS]
    print(f"\n=== {title} ===")
    print(t.round(0).astype(int).to_string())
    return t


t_aic = show(
    "dAIC", "Delta AIC (all - socioeconomic)   [more negative = mobile adds more]"
)
_ = show("dLogLik", "Delta LogLik (all - socioeconomic) [larger = mobile adds more]")

# rank windows by mobile contribution (|dAIC|), per year
print("\n=== windows ranked by mobile contribution |dAIC| (1 = largest gain) ===")
for year in YEARS:
    order = (
        delta[delta.year == year]
        .assign(gain=lambda d: -d.dAIC)
        .sort_values("gain", ascending=False)
    )
    rank = [f"{WINDOW_LABELS[w]}" for w in order["window"]]
    print(f"  {year}: " + "  >  ".join(rank))

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
x = np.arange(len(WINDOWS))
bw = 0.38
for k, year in enumerate(YEARS):
    gain = [val(year, w, SOCIO, "AIC") - val(year, w, ALL, "AIC") for w in WINDOWS]
    ax.bar(x + (k - 0.5) * bw, gain, bw, label=year)
    for xi, v in zip(x + (k - 0.5) * bw, gain):
        ax.text(xi, v, f"{int(round(v))}", ha="center", va="bottom", fontsize=8)
ax.axvspan(
    -0.5, 0.5, color="gold", alpha=0.12
)  # highlight the residential window we keep
ax.text(
    0, ax.get_ylim()[1] * 0.02, "chosen", ha="center", fontsize=8, color="goldenrod"
)
ax.set_xticks(x)
ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], rotation=18, ha="right")
ax.set_ylabel("AIC gain from mobile services\n(socioeconomic AIC - all AIC)")
ax.set_title("Mobile-service contribution to the Dirichlet model by time window")
ax.legend(title="election")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/sensitivity_dirichlet_dAIC.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Correlation (rho) view of the same sensitivity, in the body's metric.
# rho = sqrt(adjusted R^2), per party, computed as in nb 11.5:
#   r2(observed %, fitted share*100) -> adjusted by param count, clipped at 0;
#   the reference (last) component is dropped.
# Relative gain uses the body's definition = MEAN OF PER-PARTY RELATIVES,
#   mean_p[(rho_all - rho_socio)/rho_socio],  NOT the relative of the means
#   (the two differ when a party has a small socioeconomic baseline).
# In 2019, Envie d'Europe has a very low baseline (rho_socio ~0.21) and dominates
# the mean of relatives, so we report 2019 both with and without it (as the main
# text does); 2024 has no such outlier.
# ---------------------------------------------------------------------------
ALL_M, SOC_M = "income_unemployment_pop_apps", "income_unemployment_pop"


def _r2(obs, pred):
    return 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2)


def _k(year, model):  # adj-R^2 penalty: subtract one intercept per modelled component
    return pd.read_csv(f"{DATA_DIR}/dirichlet/{year}/df_coefficients_{model}.csv").shape[0] - (7 if year == "2019" else 8)  # 2019 has 7 components, 2024 has 8


def rho_per_party(year, window, model):
    inp = pd.read_csv(f"{DATA_DIR}/dirichlet/df_data_europe_{year}_{window}.csv")
    obs = inp[[c for c in inp.columns if c.endswith("_votes")]]
    pred = pd.read_csv(f"{SENS_DIR}/{year}/{window}/df_prediction_{model}.csv") * 100  # shares -> %
    k, n, rhos = _k(year, model), pred.shape[0], []
    for j in range(obs.shape[1] - 1):  # drop last component (reference), as in 11.5
        adj = 1 - ((1 - _r2(obs.iloc[:, j].values, pred.iloc[:, j].values)) * (n - 1)) / (n - k - 1)
        rhos.append(np.sqrt(max(adj, 0.0)))
    return np.array(rhos)


rows_rho = []
for y in YEARS:
    for w in WINDOWS:
        a = rho_per_party(y, w, ALL_M)
        s = rho_per_party(y, w, SOC_M)
        rel = (a - s) / s                    # per-party relative gain
        drop = int(np.argmin(s))             # lowest socioeconomic-baseline party (Envie d'Europe in 2019)
        rows_rho.append({
            "year": y, "window": w,
            "rho_socio": s.mean(), "rho_all": a.mean(),
            "d_rho": a.mean() - s.mean(),                          # absolute gain in mean rho
            "rel_gain_%": rel.mean() * 100,                        # mean of relatives (body metric)
            "rel_gain_excl_%": np.delete(rel, drop).mean() * 100,  # excl. lowest-baseline party
        })
rho = pd.DataFrame(rows_rho)


def show_rho(metric, title, nd=4):
    t = rho.pivot_table(index="window", columns="year", values=metric).reindex(WINDOWS)
    t.index = [WINDOW_LABELS[w] for w in WINDOWS]
    print(f"\n=== {title} ===\n{t.round(nd).to_string()}")


show_rho("rho_all", "Mean correlation rho, full model, per window")
show_rho("d_rho", "Absolute gain in mean rho (all - socioeconomic)")
show_rho("rel_gain_%", "Relative gain = mean of per-party relatives (%)  [body metric]", nd=2)
show_rho("rel_gain_excl_%", "Relative gain excl. lowest-baseline party (%)  [2019 drops Envie d'Europe; 2024 unused]", nd=2)
print(f"\nSocioeconomic baseline rho (window-invariant): "
      f"2019={rho[rho.year=='2019'].rho_socio.iloc[0]:.4f}  "
      f"2024={rho[rho.year=='2024'].rho_socio.iloc[0]:.4f}")
print("[note] residential (20-07) is the smallest gain on every metric (conservative); "
      "socio rho identical across windows. 2019: report rel_gain both with (rel_gain_%) "
      "and without (rel_gain_excl_%) Envie d'Europe; 2024: use rel_gain_%.")

## Reading the temporal-sensitivity result

**Metric.** For each weekday time window we refit the full Dirichlet model (`all` = socioeconomic + mobile) and the socioeconomic-only baseline, and report the gain in likelihood criteria (the same LogLik/AIC/BIC used in the main Appendix). The socioeconomic baseline is window-invariant (census data has no hour), so the gain isolates the mobile-service contribution.

**What we find.**
1. **Mobile services add large, significant explanatory power in *every* window** (ΔAIC strongly negative throughout: −6.9k to −8.4k in 2019, −14.2k to −15.7k in 2024). The headline finding does not depend on the hour choice.
2. **The residential window (20-07) gives among the *smallest* gains; daytime/full give the largest; the morning window is among the weakest.** So our choice is *conservative*: other windows would show an even larger mobile contribution. The reviewer's concern — that the evening/night window inflates our result — is the opposite of what we observe.
3. The morning hours the reviewer flagged for news (`07-13`) are among the weakest, not strongest: including the morning peak does not improve fit.

**Why we keep 20-07 anyway — validity, not fit.** Evening/night traffic reflects residents *at home*, matching where they vote; daytime windows mix in commuters/workers/visitors, i.e. workplace geography, a different population. We retain the residential window on this validity ground, accepting a slightly smaller (lower-bound) measured gain.

**Consistency check.** The `20-07` socioeconomic and intercept models reproduce the main Table exactly; the app-based models differ by ~6% because this sensitivity uses a clean half-open `[20:00,07:00)` partition, whereas the main analysis includes the 07:00 hour. The main results stand.


In [ ]:
(0.91+0.70+0.69+0.77+0.71+0.49+0.80)/7

In [ ]:
(0.70+0.50+0.42+0.73+0.69+0.42+0.58)/7

In [ ]:
(0.724 - 0.57714)/0.57714 * 100